# TweetAge · 04 Submission v2

Genera el CSV de una de las especificaciones guardadas por `02_experimentos` (secciones 12–14):

| `SPEC` | Qué es | Δ confirmación |
|---|---|---|
| `greedy_ent` | sección 14: char4/5/6 sin casefold + TweetTokenizer + shape7 + char_seq4 + LM de entidades | +0.0068 |
| `greedy_conservador_ent` | sección 14: char4/5 sin casefold + TweetTokenizer + LM de entidades | +0.0060 |
| `greedy` | sección 13: camino completo sin entidades | +0.0051 |
| `greedy_conservador` | sección 13: camino conservador sin entidades (sin parámetros ajustados) | +0.0044 |
| `final`, `mezcla_B`, `conservadora` | sección 12: combinación automática (no supera a la mezcla vigente; solo registro) | ≤ 0 |

Cada canal se entrena con **todo** `age_train.csv` (aplicando la variante de preprocesamiento de la spec) y predice `age_test.csv`. Las probabilidades quedan en caché, así que generar otra spec con los mismos canales es casi instantáneo.

In [ ]:
import time

import numpy as np
import pandas as pd

from tweetage import blend_opt as B
from tweetage import channels_extra as CX
from tweetage import evaluation as E
from tweetage import experiments_v2 as XV
from tweetage import paths, registry
from tweetage import pipeline as P
from tweetage import preprocess as PP

SPEC = "greedy_kn   "  # "greedy_ent" | "greedy_conservador_ent" | "greedy" | "greedy_conservador" | ...
N_JOBS = 5

spec = XV.load_spec(SPEC)
print(f"Spec '{SPEC}': F1 confirmación {spec['f1_confirmacion']:.4f} (Δ {spec['delta_confirmacion']:+.4f})")
print("Variante:", spec["flags"])
print("Reemplazos:", spec.get("swaps"), "| agregados:", [a["name"] for a in spec.get("adds") or []])

In [ ]:
train, test = paths.load_data()
X, X_test = paths.texts(train), paths.texts(test)
y = train["age_range"].to_numpy()

flags = spec["flags"]
Xtr, ytr = XV.apply_variant_train(X, y, flags)
Xte = XV.apply_variant_eval(X_test, flags)
print(f"Train usado: {len(Xtr):,} de {len(X):,} filas | test: {len(Xte):,}")

t0 = time.perf_counter()
resolved = CX.resolve(XV.spec_channels(spec), flags)
probs_real = CX.cached_probs(
    list(resolved.values()), Xtr, ytr, Xte, tag=f"full_{CX.variant_name(flags)}", n_jobs=N_JOBS
)
probs = {logical: probs_real[real] for logical, real in resolved.items()}
masks = PP.gate_masks(Xte)
print(f"Canales listos en {time.perf_counter() - t0:.0f} s")

In [ ]:
scores = XV.predict_spec(spec, probs, masks)
predictions = np.asarray(P.CLASES)[scores.argmax(axis=1)]

# Referencia: la mezcla vigente (0.60021) sobre el mismo test
p_vigente = P.blend(
    CX.cached_probs(CX.BASE_NAMES, X, y, X_test, tag="full_base", n_jobs=N_JOBS),
    PP.gate_masks(X_test)["mention"],
)
print(f"Predicciones distintas a la mezcla vigente: {np.mean(p_vigente.argmax(1) != scores.argmax(1)):.2%}")
pd.DataFrame({"vigente": E.pred_distribution(p_vigente), SPEC: E.pred_distribution(scores)}).T

In [ ]:
exp_id = registry.log(
    f"submission_v2_{SPEC}",
    spec,
    "full",
    spec["f1_confirmacion"],
    float("nan"),
    notas=f"submission v2 ({SPEC}); f1 = confirmación local",
)
path = paths.save_submission(test, predictions, exp_id, f"v2_{SPEC}")
print("Archivo listo:", path)
print("Tras subirlo: registry.set_kaggle_score(", exp_id, ", <score público>)")